# GPT Homework: Parallel Attention and Addition

This notebook tracks two Karpathy GPT homework exercises:

- **EX1:** merge the separate `Head` and `MultiHeadAttention` classes so all attention heads are computed together with tensor operations.
- **EX2:** train a small decoder-only Transformer on synthetic addition problems, using examples like `a+b=c` as the language modeling task.

The goal is to understand the Transformer mechanics directly: token/position embeddings, causal self-attention, multi-head attention, feed-forward blocks, residual connections, layer norm, loss masking, and autoregressive generation.


In [1]:
import math
import inspect
from dataclasses import dataclass
import random
import os
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:

# Check for XLA (TPU), then CUDA (GPU), then CPU
if 'COLAB_TPU_ADDR' in os.environ:
    # Only import torch_xla when running on TPU
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
elif torch.cuda.is_available():
    device = 'cuda'
elif torch.mps.is_available():
  device = 'mps'
else:
    device = 'cpu'
print(device)

mps


In [3]:
# hyperparameters
batch_size = 16 # B how many independent sequences will we process in parallel?
block_size = 12 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
g = torch.Generator().manual_seed(4242)


In [4]:
chars = sorted(['+','0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=',' '])
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [5]:
decimal_place = 1000 # 10 is single digit, 100 is 2, 1000 is 3 etc
n_data = int(math.pow(decimal_place,2)) # 100 x 100 = 10000 total permutations
n_val = int(0.1 * n_data)


pairs = []
for a in range(decimal_place):
  for b in range(decimal_place):
    pairs.append((a,b))

random.shuffle(pairs)

val_pairs = pairs[:n_val]
train_pairs = pairs[n_val:]

In [6]:
def get_batch(split):
    """
    returns a batch of consiting of batch_size chunks/sequences.
    each chunk is block_size long, which means it contains block_size tokens.
    in Y, we are masking the tokens we dont want the model to learn to predict (operands)
    """
    batch_pairs = train_pairs if split == 'train' else val_pairs
    ix = torch.randint(len(batch_pairs), (batch_size,)).tolist()

    rows = []
    for i in ix:
        a, b = batch_pairs[i]
        c = a + b
        c_str = f"{c:04d}"[::-1]
        s = f"{a:03d}+{b:03d}={c_str} "
        rows.append(torch.tensor(encode(s)))
    full = torch.stack(rows)                 # (B, 9)
    x = full[:, :block_size]                  # (B, 8)
    y = full[:, 1:block_size+1].clone()
    y[:, 0:7] = -1

    x, y = x.to(device), y.to(device)
    return x, y

In [9]:
@torch.no_grad()
def estimate_loss():
    """
    returns the model's current loss calculated using 'eval_iters' number of batches, using model(x,y) calls and calls model.train()
    """
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self,n_head,head_size):
        super().__init__()

        # self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)


        # super().__init__()
        self.key = nn.Linear(n_embd, n_head * head_size, bias=False)
        self.query = nn.Linear(n_embd, n_head * head_size, bias=False)  # 64,16
        self.value = nn.Linear(n_embd, n_head * head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #??
        # self.dropout = nn.Dropout(dropout) # ?
        # print(head_size)
        self.n_head = n_head
        self.head_size = head_size

        # print(self.head_size)


    def forward(self, x):

        n_head = self.n_head
        head_size = self.head_size

        B,T,C = x.shape #16,12,64

        k = self.key(x)   # (B,T,C)

        q = self.query(x) # (B,T,C)

        v = self.value(x) # (B,T,C)

        k = k.view(B,T,n_head,head_size)
        q = q.view(B,T,n_head,head_size)
        v = v.view(B,T,n_head,head_size)

        # (B,n_head,T, head_size)
        k = k.transpose(-3,-2)
        q = q.transpose(-3,-2)
        v = v.transpose(-3,-2)


        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * head_size**-0.5 # (B,n_head,T, head_size) @ (B,n_head, head_size,T) -> (B, n_head, T, T)

        # wei now contains the 'affinities' between q and k, for example if the query is something like I am looking for an verb, then the key with
        # highest affinity will be the one that contains a verb.

        # print(wei)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, n_head, T, T)
        # masking because we are not allowed to have future info aggregated from future tokens (those comming from y)
        # this is what makes this decoder.
        # print(wei)
        wei = F.softmax(wei, dim=-1) # (B, n_head, T, T)
        # exp(-inf) = 0

        wei = self.dropout(wei)

        # perform the weighted aggregation of the values

        out = wei @ v # (B, n_head, T, T) @ (B,n_head,T, head_size) -> (B, n_head, T, head_size)
        # we aggregate over v not x directly.

        out = out.transpose(-3,-2) #(B, T,n_head,  head_size)
        out = out.reshape(B,T, n_head * head_size) #B,T, n_head * head_size

        out = self.dropout(self.proj(out))

        # print(f'out shape{out.shape}') 16, 12, 64

        return out



class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),)

    def forward(self, x):
        # x = B,T, n_head * head_size = 16, 12, 64
        # linear, x = B,T, n_head * head_size = 16, 12, 64 -> 16, 12, 256
        # relu 16, 12, 256
        # linear 16,12,256 -> 16, 12, 64
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # B,T, n_head * head_size
        # the x being added, is it for residual connections?
        x = x + self.ffwd(self.ln2(x))
        return x #16, 12, 64

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape #(B,T)
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C) -> B,
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size) #16, 12, 13

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets, ignore_index=-1)

        return logits, loss




In [10]:
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # what is the model.train in estimate_loss?
    # it just sets the model in training mode

    # max iterations vs estimate loss iterations
    # idk if we are doing mini batch training here.

    # evaluate the loss
    logits, loss = model(xb, yb) #forward pass


    optimizer.zero_grad(set_to_none=True) # setting gradients to zero, we dont want to accumalte them.

    loss.backward()
    optimizer.step()

# generate from the model
# context = torch.zeros((1, 1), dtype=torch.long, device=device)
# print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.201741 M parameters
step 0: train loss 2.6697, val loss 2.6716
step 100: train loss 1.4110, val loss 1.4104
step 200: train loss 1.3962, val loss 1.3970
step 300: train loss 1.3912, val loss 1.3922
step 400: train loss 1.3910, val loss 1.3890
step 500: train loss 1.3894, val loss 1.3905
step 600: train loss 1.3851, val loss 1.3839
step 700: train loss 1.3178, val loss 1.3188
step 800: train loss 1.1052, val loss 1.1075
step 900: train loss 0.9538, val loss 0.9531
step 1000: train loss 0.9330, val loss 0.9323
step 1100: train loss 0.9228, val loss 0.9227
step 1200: train loss 0.9163, val loss 0.9148
step 1300: train loss 0.8250, val loss 0.8230
step 1400: train loss 0.6692, val loss 0.6646
step 1500: train loss 0.6350, val loss 0.6319
step 1600: train loss 0.6026, val loss 0.6020
step 1700: train loss 0.5177, val loss 0.5176
step 1800: train loss 0.4876, val loss 0.4893
step 1900: train loss 0.4647, val loss 0.4578
step 2000: train loss 0.2624, val loss 0.2578
step 2100: train loss 0.

In [11]:
@torch.no_grad()
def generate_addition(model, a, b, max_new_tokens=5):
    model.eval()

    # prompt is only the problem, no answer
    prompt = f"{a:03d}+{b:03d}="
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)

        # greedy decoding: choose most likely next token
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next), dim=1)

    out = decode(idx[0].tolist())

    # model predicts reversed answer digits, e.g. 12+34=640 means 046 => 46
    pred_reversed = out.split("=")[1]
    pred_normal = pred_reversed[::-1]

    return {
        "prompt": prompt,
        "raw_output": out,
        "pred_reversed": pred_reversed,
        "pred_answer": int(pred_normal),
        "true_answer": a + b,
        "correct": int(pred_normal) == a + b,
    }

In [22]:
x,y = get_batch('eval')
x,y

(tensor([[ 7, 10,  5,  1,  3,  7, 10, 12,  3,  6,  9,  2],
         [ 8,  6,  6,  1,  8,  7,  8, 12,  2,  2,  5,  3],
         [ 7,  7, 10,  1,  4, 10,  3, 12, 11,  5, 10,  2],
         [ 7,  9,  5,  1,  5,  5,  6, 12,  9,  2, 11,  2],
         [ 6,  7,  5,  1,  9,  3,  7, 12, 10,  8,  3,  3],
         [ 9,  9,  4,  1,  6, 10,  6, 12,  8,  7,  4,  3],
         [ 3,  2,  3,  1,  8,  2,  9, 12, 10,  2,  9,  2],
         [ 3,  9, 10,  1,  3, 11,  3, 12, 11,  8,  5,  2],
         [ 8,  5, 11,  1, 10,  2,  7, 12,  6,  6,  6,  3],
         [ 4,  7, 11,  1, 10, 10,  7, 12,  6,  6,  3,  3],
         [ 7,  8,  4,  1, 10,  5,  7, 12,  9, 11,  5,  3],
         [ 7,  9,  3,  1,  4,  3, 11, 12,  2, 11,  9,  2],
         [ 5,  5,  6,  1, 10, 11, 10, 12,  4,  5,  4,  3],
         [ 3,  3,  7,  1,  4,  5, 11, 12,  6,  7,  5,  2],
         [ 6,  7,  8,  1,  7,  9,  6, 12,  2,  5,  2,  3],
         [11, 11,  2,  1,  3,  8,  8, 12,  8,  7,  3,  3]], device='mps:0'),
 tensor([[-1, -1, -1, -1, -1, -1, -1, 

In [23]:
x.shape

torch.Size([16, 12])

In [24]:
logits, loss = model(x, y)

In [25]:
logits.shape # 

torch.Size([192, 13])

In [29]:
logits.view(16,12,13)

tensor([[[ 3.0072e+00,  1.0409e+00,  8.5910e+00,  ..., -5.0287e-01,
          -8.4300e-01,  1.5878e+00],
         [-1.4434e+00, -3.2915e+00, -1.0346e+00,  ..., -6.4298e-01,
          -1.6936e+00, -3.3511e+00],
         [-2.9811e+00, -1.0746e+00,  4.4857e+00,  ..., -2.3699e+00,
           3.2619e+00, -3.6952e-01],
         ...,
         [ 3.5213e-01,  6.1019e-02, -6.5073e+00,  ...,  2.5501e+00,
          -1.3331e+00,  5.7838e-02],
         [-4.6217e-01, -1.1884e+00,  1.2643e+01,  ..., -2.8387e+00,
           5.1056e-02, -1.3269e+00],
         [ 1.4272e+01, -2.0677e-01, -3.3694e-01,  ..., -1.1613e+00,
          -3.4985e-01, -5.1484e-02]],

        [[ 6.6150e+00, -3.6321e-01,  1.5626e+00,  ...,  6.4555e-01,
          -1.4987e+00,  4.9088e-01],
         [-4.5133e+00, -1.3984e+00,  5.3430e-01,  ...,  5.1781e+00,
           6.3198e+00, -8.6620e-01],
         [-2.6558e+00, -8.4987e-01,  7.9696e+00,  ..., -2.7240e+00,
           3.4884e+00, -3.8969e-01],
         ...,
         [-6.1849e-01, -9

In [27]:
loss

tensor(0.0002, device='mps:0', grad_fn=<NllLossBackward0>)

In [13]:
a,b = random.randint(0,999),random.randint(0,999)
model.eval()

max_new_tokens=5

prompt = f"{a:03d}+{b:03d}="
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

for _ in range(max_new_tokens):
    idx_cond = idx[:, -block_size:]
    logits, _ = model(idx_cond)
    print(idx.shape,logits.shape)

    # greedy decoding: choose most likely next token
    logits = logits[:, -1, :]
    idx_next = torch.argmax(logits, dim=-1, keepdim=True)
    idx = torch.cat((idx, idx_next), dim=1)

out = decode(idx[0].tolist())

# model predicts reversed answer digits, e.g. 12+34=640 means 046 => 46
pred_reversed = out.split("=")[1]
pred_normal = pred_reversed[::-1]

int(pred_normal) == a + b
print(a,b,pred_normal)

torch.Size([1, 8]) torch.Size([1, 8, 13])
torch.Size([1, 9]) torch.Size([1, 9, 13])
torch.Size([1, 10]) torch.Size([1, 10, 13])
torch.Size([1, 11]) torch.Size([1, 11, 13])
torch.Size([1, 12]) torch.Size([1, 12, 13])
595 142  0737


In [ ]:
block_size

In [ ]:
a,b = random.randint(0,999),random.randint(0,999)
model.eval()

max_new_tokens=5

# prompt is only the problem, no answer
prompt = f"{a:03d}+{b:03d}="
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

idx

In [18]:
idx.shape

torch.Size([1, 8])

In [19]:
idx_cond = idx[:, -block_size:]
idx_cond.shape

torch.Size([1, 8])

In [20]:
logits, _ = model(idx_cond)
logits.shape # B, T, vocab_size

torch.Size([1, 8, 13])

In [21]:
logits = logits[:, -1, :]

In [22]:
logits

tensor([[-1.0157, -1.0842, -0.6247, -4.0935, -1.5970, -0.4713,  0.2536, 11.6773,
         -0.0339,  0.3695, -5.2691,  1.2154, -0.9047]],
       grad_fn=<SelectBackward0>)

In [23]:
logits.shape

torch.Size([1, 13])

In [24]:
idx_next = torch.argmax(logits, dim=-1, keepdim=True)
idx_next

tensor([[7]])

In [25]:
idx = torch.cat((idx, idx_next), dim=1)
idx

tensor([[10, 11,  4,  1,  6,  2,  5, 12,  7]])

In [26]:
decode(idx[0].tolist())

'892+403=5'

In [ ]:
#961+767=5

## EX3: Kimi Delta Attention (KDA) on MLX

**Feasibility verdict: yes on both requested platforms.** This section uses **MLX + Metal on Apple silicon**. For TPU, `sglang-jax` contains both a step-by-step JAX reference and a chunked Pallas KDA forward kernel. The TPU/Pallas path is currently inference-oriented; optimized TPU training would still need a matching custom VJP/backward. A community MLX K3 port also demonstrates end-to-end KDA on Apple silicon.

The implementation below follows the [Kimi K3 technical report](https://github.com/MoonshotAI/Kimi-K3/blob/main/k3_tech_report.pdf): kernel-4 causal depthwise convolutions on Q/K/V, L2-normalized Q/K, the delta-rule recurrent state, K3's lower-bounded log-decay `g = -5 * sigmoid(exp(A_log) * z)`, head-wise RMSNorm, and a full-rank output gate. MLX training now uses a custom VJP backed by fused Metal forward and backward kernels. It checkpoints the recurrent state every 16 tokens, recomputes one chunk on-chip during backpropagation, and avoids materializing every token's state in global memory. A leaner fused forward kernel is selected automatically for evaluation/inference.

Important scope note: production K3 is **hybrid**, repeating three KDA layers followed by one global Gated-MLA layer. `KDAAdditionModel` intentionally uses KDA in every block so we can isolate the requested replacement on this tiny addition task. It also uses NoPE, as K3 does, because recurrence and causal convolutions carry order information.

Primary references: [Kimi K3 report](https://github.com/MoonshotAI/Kimi-K3), [Kimi Linear paper](https://arxiv.org/abs/2510.26692), [official Kimi Linear repository](https://github.com/MoonshotAI/Kimi-Linear), [JAX/Pallas KDA](https://github.com/sgl-project/sglang-jax/tree/main/python/sgl_jax/srt/kernels/kda), and the [MLX K3 port](https://github.com/PipeNetwork/kimi-k3-mlx).


In [27]:
import mlx.core as mx
import mlx.nn as mlx_nn
import mlx.optimizers as mlx_optim
import numpy as np
import pandas as pd
import time

from karpathygpt_hw.kda_mlx import (
    KDAAdditionModel,
    benchmark_attention,
    benchmark_kda_layer_training,
    benchmark_kda_training,
    expected_speedup_table,
    gated_delta_metal,
    gated_delta_metal_trainable,
    gated_delta_reference,
    masked_addition_loss,
    parameter_count,
    platform_report,
)

# Numerical parity: differentiable recurrence versus fused Metal forward.
mx.random.seed(4242)
B, T, H, D = 2, 12, 4, 16
q = mx.random.normal((B, T, H, D))
k = mx.random.normal((B, T, H, D))
v = mx.random.normal((B, T, H, D))
retention = mx.full((B, T, H, D), 0.97)
beta = mx.full((B, T, H), 0.5)
reference = gated_delta_reference(q, k, v, retention, beta)
fused = gated_delta_metal(q, k, v, retention, beta)
mx.eval(reference, fused)
difference = mx.abs(reference - fused)

print(platform_report())
print('max absolute error:', float(mx.max(difference).item()))
print('mean relative error:', float((mx.mean(difference) / mx.mean(mx.abs(reference))).item()))

# Forward and VJP parity across the 16-token training checkpoint boundary.
mx.random.seed(4242)
shape = (2, 19, 2, 4)
gradient_inputs = (
    mx.random.normal(shape) * 0.1,
    mx.random.normal(shape) * 0.1,
    mx.random.normal(shape) * 0.1,
    mx.full(shape, 0.97),
    mx.full((2, 19, 2), 0.4),
)
loss_weights = mx.random.normal(shape)

def recurrence_loss(recurrence, *arguments):
    return mx.sum(recurrence(*arguments) * loss_weights)

reference_vg = mx.value_and_grad(
    lambda *args: recurrence_loss(gated_delta_reference, *args),
    argnums=(0, 1, 2, 3, 4),
)
fused_vg = mx.value_and_grad(
    lambda *args: recurrence_loss(gated_delta_metal_trainable, *args),
    argnums=(0, 1, 2, 3, 4),
)
reference_loss, reference_gradients = reference_vg(*gradient_inputs)
fused_loss, fused_gradients = fused_vg(*gradient_inputs)
mx.eval(reference_loss, fused_loss, reference_gradients, fused_gradients)
gradient_relative_errors = [
    mx.sqrt(mx.sum((expected - actual) ** 2))
    / (mx.sqrt(mx.sum(expected ** 2)) + 1e-8)
    for expected, actual in zip(reference_gradients, fused_gradients)
]
print('19-token loss absolute error:', float(mx.abs(reference_loss - fused_loss)))
print('maximum gradient relative L2 error:', float(mx.max(mx.stack(gradient_relative_errors))))

# Causality: changing future K/V/write inputs cannot change an earlier prefix.
causal_cutoff = 7
future_positions = mx.arange(shape[1])[None, :, None, None] > causal_cutoff
perturbed_inputs = list(gradient_inputs)
perturbed_inputs[1] = mx.where(
    future_positions, gradient_inputs[1] + 0.5 * mx.random.normal(shape), gradient_inputs[1]
)
perturbed_inputs[2] = mx.where(
    future_positions, gradient_inputs[2] + 0.5 * mx.random.normal(shape), gradient_inputs[2]
)
causal_original = gated_delta_metal_trainable(*gradient_inputs)
causal_perturbed = gated_delta_metal_trainable(*perturbed_inputs)
causal_prefix_error = mx.max(mx.abs(
    causal_original[:, :causal_cutoff + 1] - causal_perturbed[:, :causal_cutoff + 1]
))
print('causal-prefix max absolute error:', float(causal_prefix_error))

{'machine': 'arm64', 'mlx_version': '0.31.2', 'metal_available': 'True', 'default_device': 'Device(gpu, 0)', 'training_path': 'MLX checkpointed fused Metal custom VJP', 'inference_path': 'MLX fused custom Metal kernel'}
max absolute error: 0.00390625
mean relative error: 2.82661e-07
19-token loss absolute error: 7.45058e-09
maximum gradient relative L2 error: 0.00082821
causal-prefix max absolute error: 0.0


In [28]:
# The fused module selects the custom-VJP kernel in train mode and the lean
# forward-only kernel after kda_model.eval().
mx.random.seed(4242)
kda_model = KDAAdditionModel(
    vocab_size=vocab_size,
    dims=n_embd,
    num_heads=n_head,
    num_layers=n_layer,
    use_fused_kernel=True,
)
smoke_logits = kda_model(mx.array(np.asarray(get_batch('train')[0].cpu())))
mx.eval(smoke_logits)
{'logits_shape': smoke_logits.shape, 'parameters': parameter_count(kda_model)}

{'logits_shape': (16, 12, 13), 'parameters': 229725}

In [29]:
# A deterministic random 90/10 split avoids digit-pattern leakage.
rng = np.random.default_rng(4242)
val_ids = rng.choice(1_000_000, size=100_000, replace=False)
val_mask = np.zeros(1_000_000, dtype=bool)
val_mask[val_ids] = True

def make_mlx_rows(a_values, b_values):
    rows = [
        [stoi[c] for c in f"{a:03d}+{b:03d}={str(a + b).zfill(4)[::-1]} "]
        for a, b in zip(a_values, b_values)
    ]
    full = np.asarray(rows, dtype=np.int32)
    inputs = full[:, :block_size]
    targets = full[:, 1:block_size + 1].copy()
    targets[:, :7] = -1
    return mx.array(inputs), mx.array(targets)

def get_mlx_addition_batch(size=64):
    while True:
        pair_ids = rng.integers(0, 1_000_000, size * 2)
        pair_ids = pair_ids[~val_mask[pair_ids]][:size]
        if len(pair_ids) == size:
            return make_mlx_rows(pair_ids // 1000, pair_ids % 1000)

kda_model.train()
optimizer_mlx = mlx_optim.AdamW(learning_rate=1e-3, weight_decay=0.01)
loss_and_grad = mlx_nn.value_and_grad(kda_model, masked_addition_loss)

def kda_train_step(inputs, targets):
    loss, gradients = loss_and_grad(kda_model, inputs, targets)
    optimizer_mlx.update(kda_model, gradients)
    return loss

compiled_kda_step = mx.compile(
    kda_train_step,
    inputs=[kda_model.state, optimizer_mlx.state],
    outputs=[kda_model.state, optimizer_mlx.state],
)

KDA_TRAIN_STEPS = 5_000
started = time.perf_counter()
for step in range(KDA_TRAIN_STEPS):
    mlx_x, mlx_y = get_mlx_addition_batch()
    kda_loss = compiled_kda_step(mlx_x, mlx_y)
    mx.eval(kda_loss, kda_model.parameters(), optimizer_mlx.state)
    if step % 1_000 == 0 or step == KDA_TRAIN_STEPS - 1:
        print(f"step {step:4d}: loss {kda_loss.item():.4f}")
training_seconds = time.perf_counter() - started

# Evaluation mode switches to the smaller forward-only Metal kernel.
kda_model.eval()
validation_ids = rng.choice(val_ids, size=1_000, replace=False)
val_a, val_b = validation_ids // 1000, validation_ids % 1000
val_x, val_y = make_mlx_rows(val_a, val_b)
validation_loss = masked_addition_loss(kda_model, val_x, val_y)
teacher_predictions = mx.argmax(kda_model(val_x), axis=-1)
mx.eval(validation_loss, teacher_predictions)
teacher_exact = np.mean(
    np.all(np.asarray(teacher_predictions)[:, 7:11] == np.asarray(val_y)[:, 7:11], axis=1)
)

prompts = mx.array(np.asarray([
    [stoi[c] for c in f"{a:03d}+{b:03d}="] for a, b in zip(val_a, val_b)
], dtype=np.int32))
generated = prompts
for _ in range(4):
    next_token = mx.argmax(kda_model(generated)[:, -1], axis=-1, keepdims=True)
    generated = mx.concatenate([generated, next_token], axis=1)
    mx.eval(generated)
expected = np.asarray([
    [stoi[c] for c in str(a + b).zfill(4)[::-1]] for a, b in zip(val_a, val_b)
])
actual = np.asarray(generated)[:, -4:]

pd.Series({
    'steps': KDA_TRAIN_STEPS,
    'batch_size': 64,
    'parameters': parameter_count(kda_model),
    'reference_training_seconds': 32.557,  # saved pre-kernel run
    'fused_training_seconds': training_seconds,
    'end_to_end_training_speedup': 32.557 / training_seconds,
    'validation_loss': validation_loss.item(),
    'teacher_forced_exact': teacher_exact,
    'autoregressive_exact': np.mean(np.all(actual == expected, axis=1)),
    'autoregressive_digit_accuracy': np.mean(actual == expected),
    'validation_examples': len(validation_ids),
})

step    0: loss 2.8589
step 1000: loss 1.1418
step 2000: loss 0.0010
step 3000: loss 0.0004
step 4000: loss 0.0002
step 4999: loss 0.0064


steps                              5000.000000
batch_size                           64.000000
parameters                       229725.000000
reference_training_seconds            32.557000
fused_training_seconds                18.583979
end_to_end_training_speedup             1.751885
validation_loss                        0.004567
teacher_forced_exact                    1.000000
autoregressive_exact                    1.000000
autoregressive_digit_accuracy           1.000000
validation_examples                  1000.000000
dtype: float64

In [30]:
# Forward + backward recurrence benchmark, excluding the rest of the model.
# The baseline is the original differentiable MLX recurrence; both paths are compiled.
training_core_speed = pd.DataFrame(benchmark_kda_training(
    (12, 64, 256, 1_024, 2_048, 4_096),
    batch_size=4,
    dims=64,
    num_heads=4,
    warmup=1,
    iterations=2,
))
training_core_speed.round(4)

   sequence_length  autograd_reference_ms  fused_metal_ms  training_core_speedup  checkpoint_state_reduction
0               12                 1.3221          0.3603                 3.6697                        12.0
1               64                 5.4755          0.9735                 5.6248                        16.0
2              256                23.9714          2.1923                10.9344                        16.0
3             1024               114.9446          8.2978                13.8524                        16.0
4             2048               269.7010         16.0294                16.8254                        16.0
5             4096               711.5103         32.4434                21.9308                        16.0

In [31]:
# Complete KDA attention-layer training step: forward, backward, and AdamW.
# This includes projections/convolutions/gates/norm/output projection, not the model MLP.
layer_training_speed = pd.DataFrame(benchmark_kda_layer_training(
    (12, 64, 256, 1_024, 2_048, 4_096),
    batch_size=4,
    dims=64,
    num_heads=4,
    warmup=1,
    iterations=2,
))
layer_training_speed.round(4)

   sequence_length  batch_size  autograd_reference_step_ms  fused_metal_step_ms  full_layer_training_speedup
0               12           4                      1.4160               0.7054                       2.0073
1               64           4                      5.9901               1.6517                       3.6266
2              256           4                     25.3850               2.7195                       9.3344
3             1024           4                    118.5138               9.9069                      11.9628
4             2048           4                    279.8060              19.1665                      14.5987
5             4096           4                    726.7005              37.1042                      19.5854

In [32]:
# Median latency over five repeats on this M5 Max. Batch=16, d_model=64, 4 heads.
# Both rows are complete attention layers, including projections; KDA also includes
# its causal convs, decay/write/output gates, normalization, and fused recurrence.
local_speed = pd.DataFrame(benchmark_attention(
    (12, 64, 256, 1_024, 2_048, 4_096),
    batch_size=16,
    dims=64,
    num_heads=4,
    warmup=8,
    iterations=30,
))
local_speed.round(4)

   sequence_length  softmax_mha_ms  kimi_delta_ms  measured_speedup  idealized_core_speedup  attention_matrix_reduction
0               12          0.2941         0.3629            0.8104                   0.375                      0.5625
1               64          0.2593         0.4788            0.5415                   2.000                     16.0000
2              256          0.3403         0.9984            0.3409                   8.000                    256.0000
3             1024          3.3677         3.1517            1.0685                  32.000                   4096.0000
4             2048         12.4193         6.2848            1.9761                  64.000                  16384.0000
5             4096         49.4681        12.6057            3.9243                 128.000                  65536.0000

In [33]:
# Published expectations from Kimi Linear are full-model GPU results, not
# theoretical limits and not measurements of this 229K-parameter toy model.
pd.DataFrame(expected_speedup_table())

  context published_speedup                              scope
0      4K           about 1x       full-model speed on MMLU-Pro
1    128K              3.98x            full-model speed on RULER
2      1M               6.3x  time per output token versus MLA

### Reading the speed numbers

- The custom Metal VJP makes the **isolated KDA forward+backward core 3.67x faster at 12 tokens**, rising to **21.93x at 4,096 tokens**, versus the original differentiable MLX recurrence (`batch=4`). This scope excludes projections, gates, normalization, loss, and optimizer.
- The optimizer-inclusive **complete KDA attention layer** is **2.01x faster at 12 tokens** and **19.59x faster at 4,096 tokens** (`batch=4`). This includes Q/K/V projections, causal convolutions, decay/write/output gates, normalization, output projection, backward, and AdamW. It excludes the surrounding residual block/MLP.
- On the complete four-layer addition model at its real `T=12`, 5,000 steps fell from **32.56 s to 18.58 s (1.75x end-to-end)**, or about 6.51 ms to 3.72 ms per step including Python batch construction and optimizer evaluation. Embeddings, four MLPs, loss, and other unchanged work dilute the core speedup.
- Backpropagation stores one `d x d` recurrent-state checkpoint per 16-token chunk and recomputes that chunk in fast threadgroup memory. That is a **16x reduction in globally stored recurrent states** for sequences of 16 tokens or longer, while avoiding an unstable attempt to reverse the decay recurrence algebraically.
- The 19-token checkpoint-boundary check matches the framework-native loss to `7.5e-9`, with maximum gradient relative L2 error `8.3e-4`; perturbing future inputs changes the checked output prefix by exactly zero. The fused model still reaches **100% exact autoregressive accuracy** on the saved 1,000-example validation run.
- The separate forward-only comparison against softmax attention is a different question: on the real addition sequence (`T=12`), the fuller KDA layer is **0.81x as fast** as fused MHA (about **23% slower**), because there is no long context to amortize KDA's extra convolutions and gates.
- The measured crossover is around `T=1024`. At `T=2048`, the MLX KDA layer is **1.98x faster**; at `T=4096`, it is **3.92x faster**. These are attention-layer microbenchmarks on this M5 Max, not end-to-end K3 throughput.
- `idealized_core_speedup` compares leading arithmetic terms: roughly `2*T^2*d` for softmax attention versus `4*T*d^2` for the recurrent delta core. It is deliberately labeled idealized; projections, convolutions, launch overhead, serial recurrence, and memory behavior explain why measured speed is much lower.
- At inference KDA replaces a `T x T` attention-score workspace with a fixed `d x d` recurrent state. During this checkpointed training pass, saved state grows linearly as `ceil(T/16) x d x d`, rather than quadratically as `T x T`. The asymptotic memory benefit is the durable result; latency only improves after the sequence is long enough and the recurrence is fused.
- The `T=64...4096` latency rows use synthetic hidden states to measure context scaling; they are **not longer arithmetic problems**. The full four-layer model is therefore reported only at its native `T=12`; the long-context optimizer-inclusive numbers are for one complete KDA attention layer.
- Limitations: the training kernel currently supports `head_dim <= 16`, implements a first-order reverse-mode VJP (not higher-order gradients/JVP), and accumulates recurrent math in float32. Timings are local M5 Max / MLX 0.31.2 medians with compilation excluded; the 2K/4K rows intentionally use fewer iterations. They do not establish production K3 throughput or TPU training performance.
- The Kimi Linear headline numbers - about `1x` at 4K, `3.98x` at 128K, and `6.3x` TPOT at 1M - were measured on a large hybrid model with specialized GPU kernels. K3's reported **2.5x overall scaling efficiency** is a broader training/capability statement and must not be read as a 2.5x attention latency claim.
